In [ ]:
# 导入数据处理相关库
import pandas as pd  # 数据框操作和CSV文件读取
import numpy as np  # 数值计算和数组操作

# 导入数据可视化库
import matplotlib.pyplot as plt  # 基础绘图库
import seaborn as sns
from matplotlib.colors import to_rgba

# 导入机器学习相关库
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.cluster import DBSCAN
from sklearn.cluster import Birch
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')

# 导入模型评估相关库
from sklearn.metrics import (silhouette_score, davies_bouldin_score, 
                             calinski_harabasz_score, adjusted_rand_score,
                             normalized_mutual_info_score, fowlkes_mallows_score,
                             confusion_matrix, classification_report)

# 设置中文显示(避免中文乱码)
plt.rcParams['font.sans-serif'] = ['SimHei'] # 正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False # 正常显示负号

# 设置绘图样式
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

In [ ]:
# 从本地CSV文件读取数据
data = pd.read_csv('CTG.NAOMIT.csv')

print(f"数据集形状——")
print(data.shape)
print(f"\n前5行数据预览：")
print(data.head())

# 检查缺失值
print("\n缺失值检查：")
missing_values = data.isnull().sum()
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "没有缺失值")

In [ ]:
# 目标变量NSP分析
# NSP是目标变量(分类变量),表示胎儿状态
# 1 = Normal(正常), 2 = Suspect(可疑), 3 = Pathologic(病理)
print("\nNSP类别分布:")
print(data['NSP'].value_counts().sort_index())

# 计算各类别比例
nsp_counts = data['NSP'].value_counts()
print("\nNSP类别比例:")
for nsp_class in sorted(data['NSP'].unique()):
    count = nsp_counts[nsp_class]
    percentage = (count / len(data)) * 100
    print(f"类别 {nsp_class}: {count} 样本 ({percentage:.2f}%)")

# 可视化NSP分布
plt.figure(figsize=(10, 6))
nsp_counts_sorted = data['NSP'].value_counts().sort_index()
plt.bar(nsp_counts_sorted.index, nsp_counts_sorted.values, color=['green', 'orange', 'red'])
plt.xlabel('NSP Category', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('NSP Target Variable Distribution', fontsize=14, fontweight='bold')
plt.xticks([1, 2, 3], ['Normal', 'Suspect', 'Pathologic'])
# 在柱子上添加数值标签
for i, v in enumerate(nsp_counts_sorted.values):
    plt.text(nsp_counts_sorted.index[i], v + 20, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 准备聚类特征
# 需要排除的列:NSP(目标变量)、Tendency(分类变量)、CLASS(分类变量)
exclude_columns = ['NSP', 'Tendency', 'CLASS']

# 提取用于聚类的特征
feature_columns = [col for col in data.columns if col not in exclude_columns]
X = data[feature_columns].values
y_true = data['NSP'].values  # 保存真实标签用于后续评估
print(f"\n用于聚类的特征数量: {len(feature_columns)}")
print(f"\n聚类数据形状: {X.shape}")

# 数据标准化
# 使用StandardScaler进行Z-score标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
# 特征相关性分析
# 计算特征相关性矩阵
correlation_matrix = data[feature_columns].corr()
# 可视化相关性热图
plt.figure(figsize=(16, 14))
sns.heatmap(correlation_matrix, 
            cmap='coolwarm', 
            center=0,
            square=True,
            linewidths=0.5,
            cbar_kws={"shrink": 0.8},
            annot=False)  # 特征较多,不显示具体数值
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 找出高度相关的特征对(相关系数>0.8)
high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.8:
            high_corr_pairs.append((
                correlation_matrix.columns[i],
                correlation_matrix.columns[j],
                correlation_matrix.iloc[i, j]
            ))

if high_corr_pairs:
    print(f"\n发现 {len(high_corr_pairs)} 对高度相关的特征(|相关系数|>0.8):")
    for feat1, feat2, corr in high_corr_pairs[:10]:  # 显示前10对
        print(f"  {feat1} <-> {feat2}: {corr:.4f}")


In [ ]:
# 数据降维可视化(PCA)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"PCA降维完成,降维后形状: {X_pca.shape}")
print(f"第一主成分解释方差比: {pca.explained_variance_ratio_[0]:.4f}")
print(f"第二主成分解释方差比: {pca.explained_variance_ratio_[1]:.4f}")
print(f"两个主成分累计解释方差比: {pca.explained_variance_ratio_.sum():.4f}")

# 可视化PCA结果(用真实标签着色)
plt.figure(figsize=(12, 8))
colors = ['green', 'orange', 'red']
labels_map = {1: 'Normal', 2: 'Suspect', 3: 'Pathologic'}

for nsp_class in sorted(data['NSP'].unique()):
    mask = y_true == nsp_class
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], 
                c=colors[nsp_class-1], 
                label=labels_map[nsp_class],
                alpha=0.6, 
                s=30,
                edgecolors='black',
                linewidth=0.5)

plt.xlabel(f'First Principal Component ({pca.explained_variance_ratio_[0]:.2%} variance)', 
           fontsize=12)
plt.ylabel(f'Second Principal Component ({pca.explained_variance_ratio_[1]:.2%} variance)', 
           fontsize=12)
plt.title('CTG Data PCA Visualization (Colored by True NSP Labels)', 
          fontsize=14, fontweight='bold')
plt.legend(fontsize=11, loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# K-means聚类实验
optimal_k = 3
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
kmeans_labels = kmeans_final.fit_predict(X_scaled)

print(f"K-means聚类完成,K={optimal_k}")
print(f"聚类中心形状: {kmeans_final.cluster_centers_.shape}")
print(f"迭代次数: {kmeans_final.n_iter_}")

# 统计各簇的样本数量
unique, counts = np.unique(kmeans_labels, return_counts=True)
print("\n各簇样本数量:")
for cluster_id, count in zip(unique, counts):
    print(f"  簇 {cluster_id}: {count} 样本 ({count/len(kmeans_labels)*100:.2f}%)")


In [ ]:
# K-means聚类结果可视化
# 在PCA降维后的2维空间中可视化聚类结果
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 子图1: K-means聚类结果
colors_kmeans = ['blue', 'green', 'red']
for cluster_id in range(optimal_k):
    mask = kmeans_labels == cluster_id
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=colors_kmeans[cluster_id],
                   label=f'Cluster {cluster_id}',
                   alpha=0.6,
                   s=30,
                   edgecolors='black',
                   linewidth=0.5)

# 绘制聚类中心(在PCA空间中)
centers_pca = pca.transform(kmeans_final.cluster_centers_)
axes[0].scatter(centers_pca[:, 0], centers_pca[:, 1],
               c='black', marker='X', s=300, 
               edgecolors='yellow', linewidth=2,
               label='Cluster Centers', zorder=10)

axes[0].set_xlabel('First Principal Component', fontsize=12)
axes[0].set_ylabel('Second Principal Component', fontsize=12)
axes[0].set_title('K-means Clustering Result (K=3)', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11, loc='best')
axes[0].grid(True, alpha=0.3)

# 子图2: 真实标签分布(作为对比)
colors_true = ['green', 'orange', 'red']
labels_map = {1: 'Normal', 2: 'Suspect', 3: 'Pathologic'}
for nsp_class in sorted(np.unique(y_true)):
    mask = y_true == nsp_class
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=colors_true[nsp_class-1],
                   label=labels_map[nsp_class],
                   alpha=0.6,
                   s=30,
                   edgecolors='black',
                   linewidth=0.5)

axes[1].set_xlabel('First Principal Component', fontsize=12)
axes[1].set_ylabel('Second Principal Component', fontsize=12)
axes[1].set_title('True NSP Labels', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11, loc='best')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# K-means聚类性能评估
# 混淆矩阵分析
print("混淆矩阵:")

# 计算混淆矩阵
cm = confusion_matrix(y_true, kmeans_labels)
print("行:真实NSP标签 (1=Normal, 2=Suspect, 3=Pathologic)")
print("列:K-means聚类标签 (0, 1, 2)")
print(cm)

# 分析每个簇对应的主要真实类别
print("\n每个K-means簇的真实标签分布:")
for cluster_id in range(optimal_k):
    mask = kmeans_labels == cluster_id
    true_labels_in_cluster = y_true[mask]
    unique_labels, label_counts = np.unique(true_labels_in_cluster, return_counts=True)
    
    print(f"\n簇 {cluster_id} (共{len(true_labels_in_cluster)}个样本):")
    for true_label, count in zip(unique_labels, label_counts):
        percentage = count / len(true_labels_in_cluster) * 100
        label_name = labels_map[true_label]
        print(f"  {label_name} (NSP={true_label}): {count} 样本 ({percentage:.2f}%)")
        
# 其他客观评估指标
# 内部评估指标(不需要真实标签)
print("\n内部评估指标(不需要真实标签):")
print("-" * 50)
silhouette = silhouette_score(X_scaled, kmeans_labels)
davies_bouldin = davies_bouldin_score(X_scaled, kmeans_labels)
calinski_harabasz = calinski_harabasz_score(X_scaled, kmeans_labels)
print(f"轮廓系数(Silhouette Score): {silhouette:.4f}")
print(f"Davies-Bouldin指数: {davies_bouldin:.4f}")
print(f"Calinski-Harabasz指数: {calinski_harabasz:.2f}")

# 外部评估指标(使用真实标签)
print("\n外部评估指标(基于真实标签):")
print("-" * 50)
ari = adjusted_rand_score(y_true, kmeans_labels)
nmi = normalized_mutual_info_score(y_true, kmeans_labels)
fmi = fowlkes_mallows_score(y_true, kmeans_labels)
print(f"调整兰德指数(ARI): {ari:.4f}")
print(f"标准化互信息(NMI): {nmi:.4f}")
print(f"Fowlkes-Mallows指数(FMI): {fmi:.4f}")

In [ ]:
# 使用3个组件进行GMM聚类
# 创建最终的GMM模型
optimal_n_components = 3
gmm_final = GaussianMixture(n_components=optimal_n_components,
                            covariance_type='full',
                            random_state=42,
                            n_init=10)
gmm_final.fit(X_scaled)
gmm_labels = gmm_final.predict(X_scaled)

# 获取每个样本属于各组件的概率(软聚类)
gmm_probabilities = gmm_final.predict_proba(X_scaled)

print(f"GMM聚类完成,组件数={optimal_n_components}")
print(f"模型收敛: {gmm_final.converged_}")
print(f"迭代次数: {gmm_final.n_iter_}")
print(f"对数似然: {gmm_final.lower_bound_:.2f}")

# 统计各簇的样本数量
unique, counts = np.unique(gmm_labels, return_counts=True)
print("\n各簇样本数量:")
for cluster_id, count in zip(unique, counts):
    print(f"  簇 {cluster_id}: {count} 样本 ({count/len(gmm_labels)*100:.2f}%)")

# 显示样本属于各簇的平均概率
print("\n各簇的平均预测概率:")
for cluster_id in range(optimal_n_components):
    mask = gmm_labels == cluster_id
    avg_prob = gmm_probabilities[mask, cluster_id].mean()
    print(f"  簇 {cluster_id}: {avg_prob:.4f} ")

In [ ]:
# GMM聚类结果可视化
# 在PCA降维后的2维空间中可视化
fig, axes = plt.subplots(1, 2, figsize=(20, 6))

# 子图1: GMM聚类结果(硬聚类)
colors_gmm = ['blue', 'green', 'red']
for cluster_id in range(optimal_n_components):
    mask = gmm_labels == cluster_id
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=colors_gmm[cluster_id],
                   label=f'Component {cluster_id}',
                   alpha=0.6,
                   s=30,
                   edgecolors='black',
                   linewidth=0.5)
# 绘制GMM均值中心(在PCA空间中)
means_pca = pca.transform(gmm_final.means_)
axes[0].scatter(means_pca[:, 0], means_pca[:, 1],
               c='black', marker='X', s=300,
               edgecolors='yellow', linewidth=2,
               label='Component Means', zorder=10)

axes[0].set_xlabel('First Principal Component', fontsize=12)
axes[0].set_ylabel('Second Principal Component', fontsize=12)
axes[0].set_title('GMM Clustering Result (n_components=3)', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10, loc='best')
axes[0].grid(True, alpha=0.3)

# 子图2: 真实标签分布(作为对比)
colors_true = ['green', 'orange', 'red']
labels_map = {1: 'Normal', 2: 'Suspect', 3: 'Pathologic'}
for nsp_class in sorted(np.unique(y_true)):
    mask = y_true == nsp_class
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=colors_true[nsp_class-1],
                   label=labels_map[nsp_class],
                   alpha=0.6,
                   s=30,
                   edgecolors='black',
                   linewidth=0.5)

axes[1].set_xlabel('First Principal Component', fontsize=12)
axes[1].set_ylabel('Second Principal Component', fontsize=12)
axes[1].set_title('True NSP Labels', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10, loc='best')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# GMM软聚类可视化(根据最大概率着色,透明度表示概率)
# 设置画布
plt.figure(figsize=(10, 7))
# 准备颜色映射
colors_gmm = ['blue', 'green', 'red']

# 为了让透明度更有意义，采用批量绘制，并去掉边框
for cluster_id in range(optimal_n_components):
    # 找到属于该簇的点
    mask = gmm_labels == cluster_id
    cluster_probs = gmm_probabilities[mask, cluster_id]
    
    # 将基础颜色转为RGBA然后修改A（Alpha）通道
    base_color = to_rgba(colors_gmm[cluster_id])
    rgba_colors = np.tile(base_color, (len(cluster_probs), 1))
    rgba_colors[:, 3] = cluster_probs ** 2  # 这里使用了平方来增强透明度对比
    
    # 绘制点
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=rgba_colors,
                s=40,
                edgecolors='none',
                label=f'Component {cluster_id}')

# 绘制中心点 (均值)
means_pca = pca.transform(gmm_final.means_)
plt.scatter(means_pca[:, 0], means_pca[:, 1],
            c='black', marker='X', s=300,
            edgecolors='yellow', linewidth=2,
            label='Component Means', zorder=10)

# 美化图表
plt.xlabel('First Principal Component', fontsize=12)
plt.ylabel('Second Principal Component', fontsize=12)
plt.title('GMM Soft Clustering Analysis\n(Opacity represents confidence)', 
          fontsize=14, fontweight='bold')
plt.legend(loc='best')
plt.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# GMM聚类性能评估
# 混淆矩阵
print("\n混淆矩阵:")
gmm_cm = confusion_matrix(y_true, gmm_labels)
print("行:真实NSP标签, 列:GMM聚类标签")
print(gmm_cm)

# 其他客观评估指标
print("-" * 50)
print("内部评估指标:")
gmm_silhouette = silhouette_score(X_scaled, gmm_labels)
gmm_davies_bouldin = davies_bouldin_score(X_scaled, gmm_labels)
gmm_calinski_harabasz = calinski_harabasz_score(X_scaled, gmm_labels)
print(f"轮廓系数: {gmm_silhouette:.4f}")
print(f"Davies-Bouldin指数: {gmm_davies_bouldin:.4f}")
print(f"Calinski-Harabasz指数: {gmm_calinski_harabasz:.2f}")
print(f"BIC: {gmm_final.bic(X_scaled):.2f}")
print(f"AIC: {gmm_final.aic(X_scaled):.2f}")

print("-" * 50)
print("\n外部评估指标(基于真实标签):")
gmm_ari = adjusted_rand_score(y_true, gmm_labels)
gmm_nmi = normalized_mutual_info_score(y_true, gmm_labels)
gmm_fmi = fowlkes_mallows_score(y_true, gmm_labels)
print(f"调整兰德指数(ARI): {gmm_ari:.4f}")
print(f"标准化互信息(NMI): {gmm_nmi:.4f}")
print(f"Fowlkes-Mallows指数(FMI): {gmm_fmi:.4f}")

In [ ]:
# DBSCAN算法
# 测试不同的eps和min_samples组合
eps_values = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
min_samples_values = [5, 7, 10, 15]

print("\n测试不同参数组合:")
print("-" * 80)
print(f"{'eps':<8}{'min_samples':<15}{'n_clusters':<12}{'n_noise':<12}{'noise_ratio':<15}")
print("-" * 80)

best_params = None
best_score = -1
results_list = []

for eps in eps_values:
    for min_samples in min_samples_values:
        # 运行DBSCAN
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        dbscan_labels = dbscan.fit_predict(X_scaled)
        
        # 统计聚类数量和噪声点数量
        # DBSCAN中标签为-1表示噪声点
        n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
        n_noise = list(dbscan_labels).count(-1)
        noise_ratio = n_noise / len(dbscan_labels)
        
        results_list.append({
            'eps': eps,
            'min_samples': min_samples,
            'n_clusters': n_clusters,
            'n_noise': n_noise,
            'noise_ratio': noise_ratio,
            'labels': dbscan_labels
        })
        
        print(f"{eps:<8.1f}{min_samples:<15}{n_clusters:<12}{n_noise:<12}{noise_ratio:<15.2%}")
        
        # 如果聚类数量合理(3个簇)且噪声比例不太高(<30%),计算轮廓系数
        if  n_clusters == 3 and noise_ratio < 0.3:
            # 计算轮廓系数时排除噪声点
            mask = dbscan_labels != -1
            if mask.sum() > 0:
                score = silhouette_score(X_scaled[mask], dbscan_labels[mask])
                if score > best_score:
                    best_score = score
                    best_params = {'eps': eps, 'min_samples': min_samples}

if best_params:
    print(f"\n根据轮廓系数,推荐参数: eps={best_params['eps']}, min_samples={best_params['min_samples']}")
    print(f"对应轮廓系数: {best_score:.4f}")
else:
    print("\n未找到理想的参数组合,使用eps=2.5, min_samples=5")
    best_params = {'eps': 2.5, 'min_samples': 5}

In [ ]:
# 使用最优参数进行DBSCAN聚类
# 使用最优参数
optimal_eps = best_params['eps']
optimal_min_samples = best_params['min_samples']

dbscan_final = DBSCAN(eps=optimal_eps, min_samples=optimal_min_samples)
dbscan_labels = dbscan_final.fit_predict(X_scaled)

# 统计结果
n_clusters_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise_dbscan = list(dbscan_labels).count(-1)

print(f"DBSCAN聚类完成")
print(f"参数: eps={optimal_eps}, min_samples={optimal_min_samples}")
print(f"发现簇数量: {n_clusters_dbscan}")
print(f"噪声点数量: {n_noise_dbscan} ({n_noise_dbscan/len(dbscan_labels)*100:.2f}%)")

# 统计各簇的样本数量(不包括噪声)
unique_clusters = set(dbscan_labels) - {-1}
print("\n各簇样本数量:")
for cluster_id in sorted(unique_clusters):
    count = list(dbscan_labels).count(cluster_id)
    print(f"  簇 {cluster_id}: {count} 样本 ({count/len(dbscan_labels)*100:.2f}%)")

print(f"噪声点: {n_noise_dbscan} 样本 ({n_noise_dbscan/len(dbscan_labels)*100:.2f}%)")

In [ ]:
# DBSCAN聚类结果可视化
# 在PCA降维后的2维空间中可视化DBSCAN结果

# 定义高对比度的颜色和形状
# 簇0, 1, 2 使用对比明显的颜色；噪声点使用中性的灰色
custom_colors = ['#1f77b4', '#e1ea33', "#ff7f0e"] # 经典的深蓝、橙色、深绿
custom_markers = ['^', 'H', 's']
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 子图1: DBSCAN聚类结果
# 为每个簇分配不同颜色,噪声点用灰色表示
# 排序标签，确保 -1 (Noise) 总是第一个被处理或根据逻辑处理
unique_labels = sorted(set(dbscan_labels))

for cluster_id in unique_labels:
    mask = dbscan_labels == cluster_id
    
    if cluster_id == -1:
        # 噪声点：灰色 + 'x' 形状
        axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                       c='#808080',      # 中灰色
                       label='Noise', 
                       alpha=0.4,
                       s=25, 
                       marker='x')
    else:
        # 正常簇：使用索引匹配颜色和形状
        # 使用 cluster_id % len 确保索引安全
        idx = cluster_id % len(custom_colors)
        
        axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                       c=custom_colors[idx],
                       marker=custom_markers[idx],
                       label=f'Cluster {cluster_id}',
                       alpha=0.8,         # 增加不透明度使颜色更鲜艳
                       s=50,              # 增大尺寸
                       edgecolors='black',
                       linewidth=0.6)
        
axes[0].set_xlabel('First Principal Component', fontsize=12)
axes[0].set_ylabel('Second Principal Component', fontsize=12)
axes[0].set_title(f'DBSCAN Clustering Result ({n_clusters_dbscan} clusters + noise)', 
                 fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10, loc='best', ncol=2)
axes[0].grid(True, alpha=0.3)

# 子图2: 真实标签分布(作为对比)
colors_true = ['green', 'orange', 'red']
labels_map = {1: 'Normal', 2: 'Suspect', 3: 'Pathologic'}
for nsp_class in sorted(np.unique(y_true)):
    mask = y_true == nsp_class
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=colors_true[nsp_class-1],
                   label=labels_map[nsp_class],
                   alpha=0.6,
                   s=30,
                   edgecolors='black',
                   linewidth=0.5)

axes[1].set_xlabel('First Principal Component', fontsize=12)
axes[1].set_ylabel('Second Principal Component', fontsize=12)
axes[1].set_title('True NSP Labels', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10, loc='best')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# DBSCAN聚类性能评估
# 对于DBSCAN,需要排除噪声点来计算某些指标
mask_non_noise = dbscan_labels != -1
X_scaled_non_noise = X_scaled[mask_non_noise]
dbscan_labels_non_noise = dbscan_labels[mask_non_noise]
y_true_non_noise = y_true[mask_non_noise]

# 内部评估指标
print("-" * 50)
print("内部评估指标(排除噪声点):")
if len(set(dbscan_labels_non_noise)) > 1:
    # 只有当至少有2个簇时才能计算这些指标
    dbscan_silhouette = silhouette_score(X_scaled_non_noise, dbscan_labels_non_noise)
    dbscan_davies_bouldin = davies_bouldin_score(X_scaled_non_noise, dbscan_labels_non_noise)
    dbscan_calinski_harabasz = calinski_harabasz_score(X_scaled_non_noise, dbscan_labels_non_noise)
    
    print(f"轮廓系数: {dbscan_silhouette:.4f}")
    print(f"Davies-Bouldin指数: {dbscan_davies_bouldin:.4f}")
    print(f"Calinski-Harabasz指数: {dbscan_calinski_harabasz:.2f}")
else:
    print("DBSCAN只发现了1个簇,无法计算聚类质量指标")
    
print("-" * 50)
print(f"噪声检测:")
print(f"噪声点数量: {list(dbscan_labels).count(-1)}")
print(f"噪声点比例: {list(dbscan_labels).count(-1)/len(dbscan_labels)*100:.2f}%")

print("-" * 50)
print("外部评估指标(排除噪声点,基于真实标签):")
if len(set(dbscan_labels_non_noise)) > 1:
    dbscan_ari = adjusted_rand_score(y_true_non_noise, dbscan_labels_non_noise)
    dbscan_nmi = normalized_mutual_info_score(y_true_non_noise, dbscan_labels_non_noise)
    dbscan_fmi = fowlkes_mallows_score(y_true_non_noise, dbscan_labels_non_noise)
    
    print(f"调整兰德指数(ARI): {dbscan_ari:.4f}")
    print(f"标准化互信息(NMI): {dbscan_nmi:.4f}")
    print(f"Fowlkes-Mallows指数(FMI): {dbscan_fmi:.4f}")

# 混淆矩阵(排除噪声点)
if len(set(dbscan_labels_non_noise)) > 1:
    print("-" * 50)
    print("混淆矩阵(排除噪声点):")
    dbscan_cm = confusion_matrix(y_true_non_noise, dbscan_labels_non_noise)
    print("行:真实NSP标签, 列:DBSCAN聚类标签")
    print(dbscan_cm)

# 分析噪声点对应的真实标签分布
print("-" * 50)
print("噪声点的真实标签分布:")
noise_mask = dbscan_labels == -1
y_true_noise = y_true[noise_mask]
if len(y_true_noise) > 0:
    for nsp_class in sorted(np.unique(y_true_noise)):
        count = list(y_true_noise).count(nsp_class)
        percentage = count / len(y_true_noise) * 100
        label_name = labels_map[nsp_class]
        print(f"  {label_name} (NSP={nsp_class}): {count} 样本 ({percentage:.2f}%)")
else:
    print("  没有噪声点")


In [ ]:
# Birch聚类
# 测试不同的threshold值
threshold_values = [0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5]
n_clusters_birch = 3  # 使用3个簇与其他算法保持一致

print("-" * 80)
print(f"{'threshold':<12}{'n_subclusters':<15}{'silhouette':<15}{'davies_bouldin':<15}")
print("-" * 80)

best_threshold = None
best_score_birch = -1
birch_results = []

for threshold in threshold_values:
    # 创建Birch模型
    # branching_factor: 每个节点的最大子节点数,默认50
    # n_clusters: 最终聚类数量
    # threshold: 子簇的最大半径
    birch = Birch(n_clusters=n_clusters_birch, 
                  threshold=threshold,
                  branching_factor=50)
    birch_labels = birch.fit_predict(X_scaled)
    
    # 计算评估指标
    silhouette = silhouette_score(X_scaled, birch_labels)
    davies_bouldin = davies_bouldin_score(X_scaled, birch_labels)
    
    # 获取子簇数量
    n_subclusters = len(birch.subcluster_centers_)
    
    birch_results.append({
        'threshold': threshold,
        'n_subclusters': n_subclusters,
        'silhouette': silhouette,
        'davies_bouldin': davies_bouldin,
        'labels': birch_labels
    })
    
    print(f"{threshold:<12.1f}{n_subclusters:<15}{silhouette:<15.4f}{davies_bouldin:<15.4f}")
    
    if silhouette > best_score_birch:
        best_score_birch = silhouette
        best_threshold = threshold

print(f"\n根据轮廓系数,推荐threshold: {best_threshold}")
print(f"对应轮廓系数: {best_score_birch:.4f}")

# 可视化不同threshold的效果
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 子图1: 子簇数量随threshold变化
thresholds = [r['threshold'] for r in birch_results]
subclusters = [r['n_subclusters'] for r in birch_results]
axes[0].plot(thresholds, subclusters, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Threshold', fontsize=12)
axes[0].set_ylabel('Number of Sub-clusters', fontsize=12)
axes[0].set_title('Sub-clusters vs Threshold', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# 子图2: 轮廓系数随threshold变化
silhouettes = [r['silhouette'] for r in birch_results]
axes[1].plot(thresholds, silhouettes, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Threshold', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score vs Threshold', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)
# 标记最大值
max_idx = np.argmax(silhouettes)
axes[1].plot(thresholds[max_idx], silhouettes[max_idx],
            'r*', markersize=20, label=f'Best threshold={thresholds[max_idx]}')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# 使用最优参数进行Birch聚类
# 使用最优threshold
optimal_threshold = best_threshold
birch_final = Birch(n_clusters=n_clusters_birch,
                   threshold=optimal_threshold,
                   branching_factor=50)
birch_labels = birch_final.fit_predict(X_scaled)

print(f"Birch聚类完成")
print(f"参数: n_clusters={n_clusters_birch}, threshold={optimal_threshold}")
print(f"子簇数量: {len(birch_final.subcluster_centers_)}")
print(f"子簇中心形状: {birch_final.subcluster_centers_.shape}")

# 统计各簇的样本数量
unique, counts = np.unique(birch_labels, return_counts=True)
print("\n各簇样本数量:")
for cluster_id, count in zip(unique, counts):
    print(f"  簇 {cluster_id}: {count} 样本 ({count/len(birch_labels)*100:.2f}%)")


In [ ]:
#  Birch聚类结果可视化

# 在PCA降维后的2维空间中可视化
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 子图1: Birch聚类结果
colors_birch = ['blue', 'red', 'green']
for cluster_id in range(n_clusters_birch):
    mask = birch_labels == cluster_id
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=colors_birch[cluster_id],
                   label=f'Cluster {cluster_id}',
                   alpha=0.6,
                   s=30,
                   edgecolors='black',
                   linewidth=0.5)

# 绘制子簇中心(在PCA空间中)
subcluster_centers_pca = pca.transform(birch_final.subcluster_centers_)
axes[0].scatter(subcluster_centers_pca[:, 0], subcluster_centers_pca[:, 1],
               c='black', marker='+', s=100, linewidth=2,
               label='Sub-cluster Centers', zorder=10, alpha=0.5)

axes[0].set_xlabel('First Principal Component', fontsize=12)
axes[0].set_ylabel('Second Principal Component', fontsize=12)
axes[0].set_title(f'Birch Clustering Result (n_clusters={n_clusters_birch})', 
                 fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10, loc='best')
axes[0].grid(True, alpha=0.3)

# 子图2: 真实标签分布(作为对比)
colors_true = ['green', 'orange', 'red']
labels_map = {1: 'Normal', 2: 'Suspect', 3: 'Pathologic'}
for nsp_class in sorted(np.unique(y_true)):
    mask = y_true == nsp_class
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=colors_true[nsp_class-1],
                   label=labels_map[nsp_class],
                   alpha=0.6,
                   s=30,
                   edgecolors='black',
                   linewidth=0.5)

axes[1].set_xlabel('First Principal Component', fontsize=12)
axes[1].set_ylabel('Second Principal Component', fontsize=12)
axes[1].set_title('True NSP Labels', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10, loc='best')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Birch聚类性能评估
print("-" * 50)
print("内部评估指标:")
birch_silhouette = silhouette_score(X_scaled, birch_labels)
birch_davies_bouldin = davies_bouldin_score(X_scaled, birch_labels)
birch_calinski_harabasz = calinski_harabasz_score(X_scaled, birch_labels)

print(f"轮廓系数: {birch_silhouette:.4f}")
print(f"Davies-Bouldin指数: {birch_davies_bouldin:.4f}")
print(f"Calinski-Harabasz指数: {birch_calinski_harabasz:.2f}")

print("-" * 50)
print("外部评估指标(基于真实标签):")
birch_ari = adjusted_rand_score(y_true, birch_labels)
birch_nmi = normalized_mutual_info_score(y_true, birch_labels)
birch_fmi = fowlkes_mallows_score(y_true, birch_labels)
print(f"调整兰德指数(ARI): {birch_ari:.4f}")
print(f"标准化互信息(NMI): {birch_nmi:.4f}")
print(f"Fowlkes-Mallows指数(FMI): {birch_fmi:.4f}")

# 混淆矩阵
print("-" * 50)
print("混淆矩阵:")
birch_cm = confusion_matrix(y_true, birch_labels)
print("行:真实NSP标签, 列:Birch聚类标签")
print(birch_cm)

# 分析每个簇对应的主要真实类别
print("\n每个Birch簇的真实标签分布:")
for cluster_id in range(n_clusters_birch):
    mask = birch_labels == cluster_id
    true_labels_in_cluster = y_true[mask]
    unique_labels, label_counts = np.unique(true_labels_in_cluster, return_counts=True)
    
    print(f"簇 {cluster_id} (共{len(true_labels_in_cluster)}个样本):")
    for true_label, count in zip(unique_labels, label_counts):
        percentage = count / len(true_labels_in_cluster) * 100
        label_name = labels_map[true_label]
        print(f"  {label_name} (NSP={true_label}): {count} 样本 ({percentage:.2f}%)")

In [ ]:
# 四种算法聚类结果并排可视化对比
# 创建2x2的子图布局
fig, axes = plt.subplots(2, 2, figsize=(18, 16))

# 定义颜色
colors_kmeans = ['blue', 'green', 'red']
colors_gmm = ['blue', 'green', 'red']
colors_birch = ['blue', 'green', 'red']

# 子图1: K-means结果
for cluster_id in range(3):
    mask = kmeans_labels == cluster_id
    axes[0, 0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                      c=colors_kmeans[cluster_id],
                      label=f'Cluster {cluster_id}',
                      alpha=0.6,
                      s=30,
                      edgecolors='black',
                      linewidth=0.3)
axes[0, 0].set_xlabel('First Principal Component', fontsize=11)
axes[0, 0].set_ylabel('Second Principal Component', fontsize=11)
axes[0, 0].set_title('K-means Clustering (K=3)', fontsize=12, fontweight='bold')
axes[0, 0].legend(fontsize=9, loc='best')
axes[0, 0].grid(True, alpha=0.3)

# 子图2: GMM结果
for cluster_id in range(3):
    mask = gmm_labels == cluster_id
    axes[0, 1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                      c=colors_gmm[cluster_id],
                      label=f'Component {cluster_id}',
                      alpha=0.6,
                      s=30,
                      edgecolors='black',
                      linewidth=0.3)
axes[0, 1].set_xlabel('First Principal Component', fontsize=11)
axes[0, 1].set_ylabel('Second Principal Component', fontsize=11)
axes[0, 1].set_title('GMM Clustering (n_components=3)', fontsize=12, fontweight='bold')
axes[0, 1].legend(fontsize=9, loc='best')
axes[0, 1].grid(True, alpha=0.3)

# 子图3: DBSCAN结果
n_clusters_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
colors_dbscan = plt.cm.Spectral(np.linspace(0, 1, n_clusters_dbscan))
for cluster_id in sorted(set(dbscan_labels)):
    if cluster_id == -1:
        mask = dbscan_labels == cluster_id
        axes[1, 0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                          c='gray',
                          label='Noise',
                          alpha=0.3,
                          s=20,
                          marker='x')
    else:
        mask = dbscan_labels == cluster_id
        axes[1, 0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                          c=[colors_dbscan[cluster_id]],
                          label=f'Cluster {cluster_id}',
                          alpha=0.6,
                          s=30,
                          edgecolors='black',
                          linewidth=0.3)
axes[1, 0].set_xlabel('First Principal Component', fontsize=11)
axes[1, 0].set_ylabel('Second Principal Component', fontsize=11)
axes[1, 0].set_title(f'DBSCAN Clustering ({n_clusters_dbscan} clusters + noise)', 
                    fontsize=12, fontweight='bold')
axes[1, 0].legend(fontsize=9, loc='best', ncol=2)
axes[1, 0].grid(True, alpha=0.3)

# 子图4: Birch结果
for cluster_id in range(3):
    mask = birch_labels == cluster_id
    axes[1, 1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                      c=colors_birch[cluster_id],
                      label=f'Cluster {cluster_id}',
                      alpha=0.6,
                      s=30,
                      edgecolors='black',
                      linewidth=0.3)
axes[1, 1].set_xlabel('First Principal Component', fontsize=11)
axes[1, 1].set_ylabel('Second Principal Component', fontsize=11)
axes[1, 1].set_title('Birch Clustering (n_clusters=3)', fontsize=12, fontweight='bold')
axes[1, 1].legend(fontsize=9, loc='best')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Four Clustering Algorithms Comparison', 
             fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
# 计算并整理所有算法的性能指标
# 对于DBSCAN,需要排除噪声点来计算某些指标
mask_non_noise = dbscan_labels != -1
X_scaled_non_noise = X_scaled[mask_non_noise]
dbscan_labels_non_noise = dbscan_labels[mask_non_noise]
y_true_non_noise = y_true[mask_non_noise]

# 创建性能指标字典
performance_metrics = {
    'Algorithm': ['K-means', 'GMM', 'DBSCAN', 'Birch'],
    'Silhouette Score': [],
    'Davies-Bouldin Index': [],
    'Calinski-Harabasz Index': [],
    'Adjusted Rand Index': [],
    'Normalized Mutual Info': [],
    'Fowlkes-Mallows Index': []
}

# K-means指标
performance_metrics['Silhouette Score'].append(
    silhouette_score(X_scaled, kmeans_labels))
performance_metrics['Davies-Bouldin Index'].append(
    davies_bouldin_score(X_scaled, kmeans_labels))
performance_metrics['Calinski-Harabasz Index'].append(
    calinski_harabasz_score(X_scaled, kmeans_labels))
performance_metrics['Adjusted Rand Index'].append(
    adjusted_rand_score(y_true, kmeans_labels))
performance_metrics['Normalized Mutual Info'].append(
    normalized_mutual_info_score(y_true, kmeans_labels))
performance_metrics['Fowlkes-Mallows Index'].append(
    fowlkes_mallows_score(y_true, kmeans_labels))

# GMM指标
performance_metrics['Silhouette Score'].append(
    silhouette_score(X_scaled, gmm_labels))
performance_metrics['Davies-Bouldin Index'].append(
    davies_bouldin_score(X_scaled, gmm_labels))
performance_metrics['Calinski-Harabasz Index'].append(
    calinski_harabasz_score(X_scaled, gmm_labels))
performance_metrics['Adjusted Rand Index'].append(
    adjusted_rand_score(y_true, gmm_labels))
performance_metrics['Normalized Mutual Info'].append(
    normalized_mutual_info_score(y_true, gmm_labels))
performance_metrics['Fowlkes-Mallows Index'].append(
    fowlkes_mallows_score(y_true, gmm_labels))

# DBSCAN指标(排除噪声点)
if len(set(dbscan_labels_non_noise)) > 1:
    performance_metrics['Silhouette Score'].append(
        silhouette_score(X_scaled_non_noise, dbscan_labels_non_noise))
    performance_metrics['Davies-Bouldin Index'].append(
        davies_bouldin_score(X_scaled_non_noise, dbscan_labels_non_noise))
    performance_metrics['Calinski-Harabasz Index'].append(
        calinski_harabasz_score(X_scaled_non_noise, dbscan_labels_non_noise))
    performance_metrics['Adjusted Rand Index'].append(
        adjusted_rand_score(y_true_non_noise, dbscan_labels_non_noise))
    performance_metrics['Normalized Mutual Info'].append(
        normalized_mutual_info_score(y_true_non_noise, dbscan_labels_non_noise))
    performance_metrics['Fowlkes-Mallows Index'].append(
        fowlkes_mallows_score(y_true_non_noise, dbscan_labels_non_noise))
else:
    # 如果DBSCAN只有一个簇,填充NA
    performance_metrics['Silhouette Score'].append(np.nan)
    performance_metrics['Davies-Bouldin Index'].append(np.nan)
    performance_metrics['Calinski-Harabasz Index'].append(np.nan)
    performance_metrics['Adjusted Rand Index'].append(np.nan)
    performance_metrics['Normalized Mutual Info'].append(np.nan)
    performance_metrics['Fowlkes-Mallows Index'].append(np.nan)

# Birch指标
performance_metrics['Silhouette Score'].append(
    silhouette_score(X_scaled, birch_labels))
performance_metrics['Davies-Bouldin Index'].append(
    davies_bouldin_score(X_scaled, birch_labels))
performance_metrics['Calinski-Harabasz Index'].append(
    calinski_harabasz_score(X_scaled, birch_labels))
performance_metrics['Adjusted Rand Index'].append(
    adjusted_rand_score(y_true, birch_labels))
performance_metrics['Normalized Mutual Info'].append(
    normalized_mutual_info_score(y_true, birch_labels))
performance_metrics['Fowlkes-Mallows Index'].append(
    fowlkes_mallows_score(y_true, birch_labels))

# 创建DataFrame并保存
data_performance = pd.DataFrame(performance_metrics)

print("\n四种算法性能指标对比表:")
print("="*150)
print(data_performance.to_string(index=False))
print("="*150)

# 保存为CSV文件
data_performance.to_csv('performance_comparison.csv', index=False, float_format='%.4f')
print("\n性能对比表已保存: performance_comparison.csv")

In [ ]:
# 性能指标可视化对比
# 创建性能指标对比图
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 指标列表(不包括算法名称)
metric_columns = [col for col in data_performance.columns if col != 'Algorithm']

# 为每个指标创建柱状图
for idx, metric in enumerate(metric_columns):
    row = idx // 3
    col = idx % 3
    
    # 获取数据
    values = data_performance[metric].values
    algorithms = data_performance['Algorithm'].values
    
    # 创建柱状图
    bars = axes[row, col].bar(algorithms, values, 
                              color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'],
                              alpha=0.8,
                              edgecolor='black',
                              linewidth=1.5)
    
    axes[row, col].set_ylabel(metric, fontsize=11, fontweight='bold')
    axes[row, col].set_title(metric, fontsize=12, fontweight='bold')
    axes[row, col].grid(True, alpha=0.3, axis='y')
    axes[row, col].tick_params(axis='x', rotation=0)
    
    # 在柱子上添加数值标签
    for bar in bars:
        height = bar.get_height()
        if not np.isnan(height):
            axes[row, col].text(bar.get_x() + bar.get_width()/2., height,
                              f'{height:.3f}',
                              ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # 根据指标类型设置y轴范围
    if 'Davies-Bouldin' in metric:
        # Davies-Bouldin越小越好,从0开始
        axes[row, col].set_ylim(bottom=0)
    else:
        # 其他指标越大越好,可以包含负值或从0开始
        if 'Rand' in metric:
            axes[row, col].set_ylim(-0.1, 1.1)
        else:
            axes[row, col].set_ylim(bottom=0)

plt.suptitle('Performance Metrics Comparison of Four Clustering Algorithms', 
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 生成聚类数量统计对比
# 统计每个算法的聚类分布
cluster_distribution = pd.DataFrame({
    'Algorithm': ['K-means', 'GMM', 'DBSCAN', 'Birch'],
    'Number of Clusters': [
        len(set(kmeans_labels)),
        len(set(gmm_labels)),
        len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0),
        len(set(birch_labels))
    ],
    'Noise Points': [
        0,
        0,
        list(dbscan_labels).count(-1),
        0
    ],
    'Cluster 0 Size': [
        list(kmeans_labels).count(0),
        list(gmm_labels).count(0),
        list(dbscan_labels).count(0) if 0 in dbscan_labels else 0,
        list(birch_labels).count(0)
    ],
    'Cluster 1 Size': [
        list(kmeans_labels).count(1),
        list(gmm_labels).count(1),
        list(dbscan_labels).count(1) if 1 in dbscan_labels else 0,
        list(birch_labels).count(1)
    ],
    'Cluster 2 Size': [
        list(kmeans_labels).count(2),
        list(gmm_labels).count(2),
        list(dbscan_labels).count(2) if 2 in dbscan_labels else 0,
        list(birch_labels).count(2)
    ]
})

print("\n聚类数量统计对比:")
print("="*100)
print(cluster_distribution.to_string(index=False))
print("="*100)

# 保存为CSV
cluster_distribution.to_csv('cluster_distribution.csv', index=False)
print("\n聚类数量统计已保存: cluster_distribution.csv")

# 可视化聚类大小分布
fig, ax = plt.subplots(figsize=(14, 8))

# 准备数据
x = np.arange(len(cluster_distribution))
width = 0.2

# 绘制堆叠柱状图
bars1 = ax.bar(x - 1.5*width, cluster_distribution['Cluster 0 Size'], width, 
               label='Cluster 0', color='#1f77b4', edgecolor='black', linewidth=1)
bars2 = ax.bar(x - 0.5*width, cluster_distribution['Cluster 1 Size'], width, 
               label='Cluster 1', color='#ff7f0e', edgecolor='black', linewidth=1)
bars3 = ax.bar(x + 0.5*width, cluster_distribution['Cluster 2 Size'], width, 
               label='Cluster 2', color='#2ca02c', edgecolor='black', linewidth=1)
bars4 = ax.bar(x + 1.5*width, cluster_distribution['Noise Points'], width, 
               label='Noise', color='gray', edgecolor='black', linewidth=1)

# 添加数值标签
for bars in [bars1, bars2, bars3, bars4]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{int(height)}',
                   ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xlabel('Algorithm', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Samples', fontsize=13, fontweight='bold')
ax.set_title('Cluster Size Distribution Across Algorithms', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(cluster_distribution['Algorithm'], fontsize=12)
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()